# Making zoomed-in plots for the paper
### A. Ordog, Sept 11, 2024

In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl
from matplotlib.patches import Ellipse
from radio_beam import Beam
import astropy.units as u
from matplotlib.patches import Rectangle

## Choose thresholds for PI and error in RM

In [ ]:
P_thr   = 0.1 # K
dRM_thr = 150 # rad/m^2

In [ ]:
# Read in RM, Pearson R, and standard error in RM:
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG_all = hdu_RM_CG[0].data
RM_CG     = RM_CG_all.copy()
rvalue_CG = hdu_RM_CG[2].data
stderr_CG = hdu_RM_CG[4].data
hdr       = hdu_RM_CG[0].header
wcs = WCS(hdr)

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G_all = hdu_RM_G[0].data
RM_G     = RM_G_all.copy()
rvalue_G = hdu_RM_G[2].data
stderr_G = hdu_RM_G[4].data

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C_all = hdu_RM_C[0].data
RM_C     = RM_C_all.copy()
rvalue_C = hdu_RM_C[2].data
stderr_C = hdu_RM_C[4].data

# Read in polarised intensity:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 

# Set outside of mosaics to NaN:
RM_CG[RM_CG_all==0.0]     = np.nan
rvalue_CG[RM_CG_all==0.0] = np.nan
stderr_CG[RM_CG_all==0.0] = np.nan
PI_CG[RM_CG_all==0.0]     = np.nan

RM_C[RM_C_all==0.0]     = np.nan
rvalue_C[RM_C_all==0.0] = np.nan
stderr_C[RM_C_all==0.0] = np.nan
PI_C[RM_C_all==0.0]     = np.nan

In [ ]:
RM_CG_filt = RM_CG.copy()
RM_G_filt = RM_G.copy()
RM_C_filt = RM_C.copy()
########################################
RM_CG_filt[PI_CG<P_thr] = np.nan
RM_CG_filt[stderr_CG>dRM_thr] = np.nan
RM_G_filt[PI_G<P_thr] = np.nan
RM_G_filt[stderr_G>dRM_thr] = np.nan
#RM_C_filt[PI_C<0.1] = np.nan
RM_C_filt[stderr_C>dRM_thr] = np.nan
########################################

RM = [RM_C_filt,RM_G_filt,RM_CG_filt]
PI = [PI_C,PI_G,PI_CG]

## Functions

In [ ]:
def make_the_plots(l0=60, b0=0, lwidth=4, filename='test', 
                   PImax=1, RMmax=300, contours=[0.2], src=None,
                  lonspace=1, latspace=0.5, circle=False):

    panels = ['(a)','(b)','(c)','(d)', '(e)', '(f)']
    
    bwidth = 3*lwidth/4
    
    llim=[l0+lwidth/2,l0-lwidth/2]
    blim=[b0-bwidth/2,b0+bwidth/2]

    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 24
    fig = plt.figure(figsize=(21.5,10))

    if src != None:
        src_c = SkyCoord(src[0], src[1], frame=Galactic, unit="deg")
        print(src_c)
    plt.subplots_adjust(hspace=0.05, wspace=0.05, left=0.07, right=0.9, top=0.98, bottom=0.08)

    # PI maps
    
    cmap = mpl.colormaps.get_cmap('gist_heat_r') 
    cmap.set_bad(color='grey')
    axPI1  = fig.add_subplot(231, projection=WCS(hdr).celestial)
    axPI2  = fig.add_subplot(232, projection=WCS(hdr).celestial)
    axPI3  = fig.add_subplot(233, projection=WCS(hdr).celestial)
    imPI1  = axPI1.imshow(PI[1], origin='lower', vmin=0, vmax=PImax, cmap=cmap)
    imPI2  = axPI2.imshow(PI[0], origin='lower', vmin=0, vmax=PImax, cmap=cmap)
    imPI3  = axPI3.imshow(PI[2], origin='lower', vmin=0, vmax=PImax, cmap=cmap)

    cb_axPI = fig.add_axes([0.91, 0.545, 0.02, 0.43])
    cbarPI = fig.colorbar(imPI3, cax=cb_axPI, orientation='vertical')
    cbarPI.set_label(r'PI (K)', fontsize=fs)

    # RM maps
    
    cmap = mpl.colormaps.get_cmap('RdBu_r') 
    cmap.set_bad(color='grey')
    axRM1  = fig.add_subplot(234, projection=WCS(hdr).celestial)
    axRM2  = fig.add_subplot(235, projection=WCS(hdr).celestial)
    axRM3  = fig.add_subplot(236, projection=WCS(hdr).celestial)
    imRM1  = axRM1.imshow(RM[1], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM2  = axRM2.imshow(RM[0], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM3  = axRM3.imshow(RM[2], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)

    cb_axRM = fig.add_axes([0.91, 0.085, 0.02, 0.43])
    cbarRM = fig.colorbar(imRM3, cax=cb_axRM, orientation='vertical')
    cbarRM.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    
    fig.text(0.008,0.42,'Galactic Latitude',fontsize=fs,rotation='vertical')
    fig.text(0.42,0.01,'Galactic Longitude',fontsize=fs,rotation='horizontal')

    i = 0
    for ax in [axPI1,axPI2,axPI3,axRM1,axRM2,axRM3]:
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        lon = ax.coords[0]
        lat = ax.coords[1]
        if ax in [axPI2,axPI3,axRM2,axRM3]:
            lat.set_ticklabel_visible(False)
        if ax in [axPI1,axPI2,axPI3]:
            lon.set_ticklabel_visible(False)
        ax.tick_params(axis='both', labelsize=fs)
        lon.set_major_formatter('d.d')
        lat.set_major_formatter('d.d')
        lon.set_ticks(spacing=lonspace* u.deg)  # Minor ticks every 1 degree
        lat.set_ticks(spacing=latspace* u.deg)    
        ax.set_ylabel('  ',fontsize=fs)
        ax.set_xlabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        ax.contour(PI[2], levels=contours, colors='black')

        if ax in [axPI3,axPI2,axRM3,axRM2]:
            beamw = 3/60
        else:
            beamw = 40/60
            
        cx = llim[0]-0.5
        cy = blim[0]+0.5
        rect = Rectangle((cx-0.4, cy-0.4), 0.8, 0.8, linewidth=1, 
                         edgecolor='black', facecolor='white', transform=ax.get_transform('galactic'),alpha=0.5)
        beam = Ellipse((cx, cy), width=beamw, height=beamw, angle=45,
                        edgecolor='black', facecolor='black', lw=1, transform=ax.get_transform('galactic'))
        ax.add_patch(rect)
        ax.add_patch(beam)

        cx = llim[0]-0.3*lwidth/4.2
        cy = blim[1]-0.25*lwidth/4.2
        ax.text(cx,cy, panels[i], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center', transform=ax.get_transform('galactic'))


        if circle:
            lcen = 158.35
            bcen = 0.2
            radcirc = 0.8
            circ = Ellipse((lcen, bcen), width=radcirc*2, height=radcirc*2, angle=45,
                        edgecolor='yellow', facecolor='None', lw=3, linestyle='dashed',
                        transform=ax.get_transform('galactic'))
            ax.add_patch(circ)

        if src != None: 
            ax.scatter([WCS(hdr).world_to_pixel(src_c)[0]],[WCS(hdr).world_to_pixel(src_c)[1]],
                        s=200, c='yellow', marker='X',edgecolor= "k")
        i = i+1
    
    for cbar in [cbarPI,cbarRM]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)

    
    
    plt.savefig('../plots/maps/'+filename+'.pdf')

    return

In [ ]:
make_the_plots(l0=189.5, b0=3, lwidth=4.2, filename='189_3_IC443_SNR', 
               PImax=0.5, RMmax=200, contours=[0.25], src=[188.9, 3.2])

In [ ]:
make_the_plots(l0=157.5, b0=0.0, lwidth=4.2, filename='158_0_Sh216_PN', 
               PImax=0.8, RMmax=150, contours=[0.3], src=[158.75,0.8],circle=True)

In [ ]:
make_the_plots(l0=172.5, b0=-0.5, lwidth=8.2, filename='172_n1_bowtie', 
               PImax=0.8, RMmax=150, contours=[0.2], src=[174.12, -2.87],lonspace=2,latspace=1)

In [ ]:
#make_the_plots(llim=[122.5,118.5], blim=[-0.5,2.5], filename='120_1_Tycho', PImax=0.8, RMmax=150, contours=[100])

In [ ]:
#make_the_plots(llim=[99.5,95.5], blim=[0,3], filename='98_1_canals', PImax=0.8, RMmax=150, contours=[0.3])

In [ ]:
#make_the_plots(llim=[70.5,66.5], blim=[1,4], filename='70_2_SNR', PImax=0.8, RMmax=150, contours=[0.15])

In [ ]:
#make_the_plots(l0=152.5, b0=0.5, lwidth=4.2, filename='153_0_Sh216_fake', PImax=0.8, RMmax=150, contours=[0.3], src=[152.2, 0.4])

In [ ]:
def make_Ransom_plots(l0=60, b0=0, lwidth=4, filename='test', 
                   PImax=1, RMmax=300, contours=[0.2], src=None,
                  lonspace=1, latspace=0.5):

    panels = ['(a)','(b)','(c)','(d)']

    lcen = 158.35
    bcen = 0.2
    radcirc = 0.8
    
    #bwidth = 3*lwidth/4
    bwidth = lwidth
    
    llim=[l0+lwidth/2,l0-lwidth/2]
    blim=[b0-bwidth/2,b0+bwidth/2]

    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 22
    fig = plt.figure(figsize=(12,10))

    if src != None:
        src_c = SkyCoord(src[0], src[1], frame=Galactic, unit="deg")
        print(src_c)
    plt.subplots_adjust(hspace=0.05, wspace=0.05, left=0.1, right=0.85, top=0.98, bottom=0.08)

    # PI maps
    
    cmap = mpl.colormaps.get_cmap('gist_heat_r') 
    cmap.set_bad(color='grey')
    axPI1  = fig.add_subplot(221, projection=WCS(hdr).celestial)
    axPI2  = fig.add_subplot(222, projection=WCS(hdr).celestial)
    imPI1  = axPI1.imshow(PI[0], origin='lower', vmin=0, vmax=PImax, cmap=cmap)
    imPI2  = axPI2.imshow(PI[2], origin='lower', vmin=0, vmax=PImax, cmap=cmap)

    cb_axPI = fig.add_axes([0.86, 0.544, 0.02, 0.435])
    cbarPI = fig.colorbar(imPI2, cax=cb_axPI, orientation='vertical')
    cbarPI.set_label(r'PI (K)', fontsize=fs)

    # RM maps
    
    cmap = mpl.colormaps.get_cmap('RdBu_r') 
    cmap.set_bad(color='grey')
    axRM1  = fig.add_subplot(223, projection=WCS(hdr).celestial)
    axRM2  = fig.add_subplot(224, projection=WCS(hdr).celestial)
    imRM1  = axRM1.imshow(RM[0], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM2  = axRM2.imshow(RM[2], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)

    cb_axRM = fig.add_axes([0.86, 0.084, 0.02, 0.435])
    cbarRM = fig.colorbar(imRM2, cax=cb_axRM, orientation='vertical')
    cbarRM.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    
    fig.text(0.008,0.42,'Galactic Latitude',fontsize=fs,rotation='vertical')
    fig.text(0.36,0.01,'Galactic Longitude',fontsize=fs,rotation='horizontal')

    i = 0
    for ax in [axPI1,axPI2,axRM1,axRM2]:
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        lon = ax.coords[0]
        lat = ax.coords[1]
        if ax in [axPI2,axRM2]:
            lat.set_ticklabel_visible(False)
        if ax in [axPI1,axPI2]:
            lon.set_ticklabel_visible(False)
        ax.tick_params(axis='both', labelsize=fs)
        lon.set_major_formatter('d.d')
        lat.set_major_formatter('d.d')
        lon.set_ticks(spacing=lonspace* u.deg)  # Minor ticks every 1 degree
        lat.set_ticks(spacing=latspace* u.deg)    
        ax.set_ylabel('  ',fontsize=fs)
        ax.set_xlabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        ax.contour(PI[2], levels=contours, colors='black')

        beam = Ellipse((lcen, bcen), width=radcirc*2, height=radcirc*2, angle=45,
                        edgecolor='white', facecolor='None', lw=3, linestyle='dashed',
                        transform=ax.get_transform('galactic'))
        ax.add_patch(beam)

        cx = llim[0]-0.3*lwidth/3.8
        cy = blim[1]-0.25*lwidth/3.8
        ax.text(cx,cy, panels[i], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center', transform=ax.get_transform('galactic'))

        i = i+1
    
    for cbar in [cbarPI,cbarRM]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)

    plt.savefig('../plots/maps/'+filename+'.pdf')

    return

In [ ]:
make_Ransom_plots(l0=158.71, b0=0.85, lwidth=0.4, filename='ransom_Sh216', 
                  PImax=0.8, RMmax=200, contours=[0.3], src=[158.75,0.8],
                  lonspace=0.1,latspace=0.1)

Ransom et al circle: diameter = 1.6 deg, centre (l,b) = (158.35,0.2)